In [ ]:
from pathlib import Path
import numpy as np
import librosa

# ============================================================
# Configuration
# ============================================================
audio_dir = Path("audio")
features_dir = Path("features")
features_dir.mkdir(parents=True, exist_ok=True)

sr = 16000
n_fft = 400          # FFT size
win_length = 400     # 25ms window (0.025 * 16000 = 400 samples)
hop_length = 160     # 10ms hop (0.010 * 16000 = 160 samples)
fmin = 0
fmax = sr // 2

audio_exts = {".flac"}

def get_audio_files():
    """Get sorted list of audio files."""
    return sorted([p for p in audio_dir.rglob("*") if p.suffix.lower() in audio_exts])

def compute_cmvn(features_list):
    """Compute global CMVN statistics."""
    all_frames = np.concatenate(features_list, axis=1)
    mean = all_frames.mean(axis=1, keepdims=True)
    std = all_frames.std(axis=1, keepdims=True)
    return mean, std

def apply_cmvn(feat, mean, std):
    """Apply CMVN normalization."""
    return (feat - mean) / (std + 1e-8)

def add_deltas(feat):
    """Add delta and delta-delta features."""
    delta = librosa.feature.delta(feat, order=1)
    delta2 = librosa.feature.delta(feat, order=2)
    return np.vstack([feat, delta, delta2])

print(f"Found {len(get_audio_files())} audio files")

In [ ]:
# ============================================================
# Experiment 1: MFCC (40 dims) vs Log-Mel (80 dims)
# ============================================================
# MFCC: Decorrelated, compact representation (loses spectral detail)
# Log-Mel: Preserves spectral envelope, better for neural networks
# ============================================================

exp1_mfcc_dir = features_dir / "exp1_mfcc40"
exp1_logmel_dir = features_dir / "exp1_logmel80"
exp1_mfcc_dir.mkdir(parents=True, exist_ok=True)
exp1_logmel_dir.mkdir(parents=True, exist_ok=True)

n_mels_80 = 80
n_mfcc_40 = 40

# Extract features
mfcc_features = []
logmel_features = []
file_stems = []

for p in get_audio_files():
    y, _ = librosa.load(p, sr=sr, mono=True)
    
    # Log-Mel (80 dims)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, win_length=win_length,
        hop_length=hop_length, n_mels=n_mels_80, fmin=fmin, fmax=fmax, power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    logmel_features.append(log_mel)
    
    # MFCC (40 dims) - from mel spectrogram
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=n_mfcc_40)
    mfcc_features.append(mfcc)
    
    file_stems.append(p.stem)

# CMVN for MFCC
mfcc_mean, mfcc_std = compute_cmvn(mfcc_features)
np.save(exp1_mfcc_dir / "cmvn_mean.npy", mfcc_mean)
np.save(exp1_mfcc_dir / "cmvn_std.npy", mfcc_std)

# CMVN for Log-Mel
logmel_mean, logmel_std = compute_cmvn(logmel_features)
np.save(exp1_logmel_dir / "cmvn_mean.npy", logmel_mean)
np.save(exp1_logmel_dir / "cmvn_std.npy", logmel_std)

# Save normalized features
for stem, mfcc, logmel in zip(file_stems, mfcc_features, logmel_features):
    np.save(exp1_mfcc_dir / f"{stem}.npy", apply_cmvn(mfcc, mfcc_mean, mfcc_std))
    np.save(exp1_logmel_dir / f"{stem}.npy", apply_cmvn(logmel, logmel_mean, logmel_std))

print(f"Exp 1: MFCC-40 shape: {mfcc_features[0].shape}, Log-Mel-80 shape: {logmel_features[0].shape}")
print(f"  - MFCC-40 saved to: {exp1_mfcc_dir}")
print(f"  - Log-Mel-80 saved to: {exp1_logmel_dir}")

In [ ]:
# ============================================================
# Experiment 2: Log-Mel vs Log-Mel + Δ + ΔΔ
# ============================================================
# Delta features capture temporal dynamics (formant transitions)
# Helps with: /r/ vs /l/, voicing onset timing, nasal transitions
# Often reduces deletions by providing velocity/acceleration cues
# ============================================================

exp2_static_dir = features_dir / "exp2_logmel_static"
exp2_delta_dir = features_dir / "exp2_logmel_delta"
exp2_static_dir.mkdir(parents=True, exist_ok=True)
exp2_delta_dir.mkdir(parents=True, exist_ok=True)

# Reuse logmel_features from Exp 1
static_features = logmel_features  # 80 dims
delta_features = [add_deltas(lm) for lm in logmel_features]  # 240 dims (80 + 80 + 80)

# CMVN for static (same as exp1_logmel)
np.save(exp2_static_dir / "cmvn_mean.npy", logmel_mean)
np.save(exp2_static_dir / "cmvn_std.npy", logmel_std)

# CMVN for delta features
delta_mean, delta_std = compute_cmvn(delta_features)
np.save(exp2_delta_dir / "cmvn_mean.npy", delta_mean)
np.save(exp2_delta_dir / "cmvn_std.npy", delta_std)

# Save normalized features
for stem, static, delta in zip(file_stems, static_features, delta_features):
    np.save(exp2_static_dir / f"{stem}.npy", apply_cmvn(static, logmel_mean, logmel_std))
    np.save(exp2_delta_dir / f"{stem}.npy", apply_cmvn(delta, delta_mean, delta_std))

print(f"Exp 2: Static shape: {static_features[0].shape}, +Delta shape: {delta_features[0].shape}")
print(f"  - Static saved to: {exp2_static_dir}")
print(f"  - +Delta saved to: {exp2_delta_dir}")

In [ ]:
# ============================================================
# Experiment 3: Number of Mel Bins (40 vs 80 vs 128)
# ============================================================
# Higher bins = better frequency resolution for fricatives (/s/ vs /ʃ/)
# Lower bins = more compact, faster compute
# 80 is standard balance; 128 helps with fine phonetic distinctions
# ============================================================

mel_configs = [40, 80, 128]
exp3_features = {n: [] for n in mel_configs}
exp3_dirs = {}

for n_mel in mel_configs:
    exp3_dirs[n_mel] = features_dir / f"exp3_logmel{n_mel}"
    exp3_dirs[n_mel].mkdir(parents=True, exist_ok=True)

# Extract features for each mel bin configuration
for p in get_audio_files():
    y, _ = librosa.load(p, sr=sr, mono=True)
    
    for n_mel in mel_configs:
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=n_fft, win_length=win_length,
            hop_length=hop_length, n_mels=n_mel, fmin=fmin, fmax=fmax, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        exp3_features[n_mel].append(log_mel)

# Compute CMVN and save for each configuration
exp3_stats = {}
for n_mel in mel_configs:
    mean, std = compute_cmvn(exp3_features[n_mel])
    exp3_stats[n_mel] = (mean, std)
    np.save(exp3_dirs[n_mel] / "cmvn_mean.npy", mean)
    np.save(exp3_dirs[n_mel] / "cmvn_std.npy", std)
    
    for stem, feat in zip(file_stems, exp3_features[n_mel]):
        np.save(exp3_dirs[n_mel] / f"{stem}.npy", apply_cmvn(feat, mean, std))

print("Exp 3: Mel bin comparison")
for n_mel in mel_configs:
    print(f"  - {n_mel} bins: shape={exp3_features[n_mel][0].shape}, saved to {exp3_dirs[n_mel]}")

## Feature Study Summary

| Experiment | Conditions | Key Metric | Expected Finding |
|------------|-----------|------------|------------------|
| **Exp 1** | MFCC-40 vs Log-Mel-80 | CER | Log-Mel preserves spectral detail, better for neural nets |
| **Exp 2** | Log-Mel vs Log-Mel+Δ+ΔΔ | CER (I/D/S breakdown) | Delta reduces **deletions** via temporal dynamics |
| **Exp 3** | 40 vs 80 vs 128 Mel bins | CER | Higher bins help fricative discrimination |

---

### IPA Distinctions & Acoustic Cues

| IPA Contrast | Acoustic Cue | Feature Importance |
|--------------|--------------|-------------------|
| /i/ vs /u/ | F1/F2 spacing | Mel resolution critical |
| /s/ vs /ʃ/ | High frequency band | Higher Mel bins helpful |
| /b/ vs /p/ | Voicing onset time | Temporal resolution (delta) |
| /r/ vs /l/ | Formant transitions | Delta / LSTM context |
| Nasal vs oral | Formant damping | Spectral envelope |

---

### Output Directories
```
features/
├── exp1_mfcc40/        # MFCC 40-dim
├── exp1_logmel80/      # Log-Mel 80-dim
├── exp2_logmel_static/ # Log-Mel (no deltas)
├── exp2_logmel_delta/  # Log-Mel + Δ + ΔΔ (240-dim)
├── exp3_logmel40/      # 40 Mel bins
├── exp3_logmel80/      # 80 Mel bins
└── exp3_logmel128/     # 128 Mel bins
```

In [ ]:
# ============================================================
# BiLSTM-CTC Model for Phoneme Recognition
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Load Transcripts & Build Vocabulary
# ============================================================

def load_transcripts(jsonl_path):
    """Load transcripts from JSONL file."""
    data = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append(item)
    return data

# Load training transcripts
train_data = load_transcripts("train_phon_transcripts.jsonl")
print(f"Loaded {len(train_data)} transcripts")
print(f"Sample: {train_data[0]}")

# Build vocabulary from IPA characters (phonetic_text field)
all_chars = set()
for item in train_data:
    text = item.get('phonetic_text', item.get('text', ''))
    all_chars.update(text)

# Sort for reproducibility, reserve 0 for CTC blank
vocab = sorted(all_chars)
char2idx = {c: i + 1 for i, c in enumerate(vocab)}  # 0 = blank
idx2char = {i + 1: c for i, c in enumerate(vocab)}
idx2char[0] = '<blank>'
vocab_size = len(vocab) + 1  # +1 for blank

print(f"Vocabulary size: {vocab_size} (including blank)")
print(f"Characters: {''.join(vocab)}")

In [ ]:
# ============================================================
# Dataset Class
# ============================================================

class SpeechDataset(Dataset):
    """Dataset for speech features and transcripts."""
    
    def __init__(self, transcripts, feature_dir, char2idx):
        self.transcripts = transcripts
        self.feature_dir = Path(feature_dir)
        self.char2idx = char2idx
        
        # Filter to only include files that exist
        self.valid_items = []
        for item in transcripts:
            # Get audio ID from utterance_id or audio_path
            audio_id = item.get('utterance_id', '')
            if not audio_id:
                audio_path = item.get('audio_path', item.get('audio', ''))
                audio_id = Path(audio_path).stem if audio_path else ''
            
            if not audio_id:
                continue
                
            feat_path = self.feature_dir / f"{audio_id}.npy"
            if feat_path.exists():
                # Get phonetic text
                text = item.get('phonetic_text', item.get('text', item.get('transcript', '')))
                self.valid_items.append((audio_id, text))
        
        print(f"Dataset: {len(self.valid_items)}/{len(transcripts)} samples found in {feature_dir}")
    
    def __len__(self):
        return len(self.valid_items)
    
    def __getitem__(self, idx):
        audio_id, text = self.valid_items[idx]
        
        # Load features (n_features, time) -> (time, n_features)
        feat = np.load(self.feature_dir / f"{audio_id}.npy").T
        feat = torch.FloatTensor(feat)
        
        # Encode text
        target = torch.LongTensor([self.char2idx[c] for c in text if c in self.char2idx])
        
        return feat, target, audio_id

def collate_fn(batch):
    """Collate function for variable-length sequences."""
    feats, targets, audio_ids = zip(*batch)
    
    # Get lengths
    feat_lengths = torch.LongTensor([len(f) for f in feats])
    target_lengths = torch.LongTensor([len(t) for t in targets])
    
    # Pad sequences
    feats_padded = pad_sequence(feats, batch_first=True)  # (B, T, F)
    targets_padded = pad_sequence(targets, batch_first=True)  # (B, S)
    
    return feats_padded, targets_padded, feat_lengths, target_lengths, audio_ids

In [ ]:
# ============================================================
# BiLSTM-CTC Model
# ============================================================
# - 2-3 BiLSTM layers
# - Hidden size: 256-512
# - Dropout: 0.2-0.3
# - Linear -> vocab_size + 1 (blank)
# ============================================================

class BiLSTM_CTC(nn.Module):
    """BiLSTM model for CTC-based speech recognition."""
    
    def __init__(self, input_dim, hidden_dim=256, num_layers=3, dropout=0.3, vocab_size=None):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # BiLSTM output is hidden_dim * 2 (forward + backward)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
    
    def forward(self, x, lengths):
        """
        Args:
            x: (batch, time, features)
            lengths: (batch,) actual lengths before padding
        Returns:
            log_probs: (time, batch, vocab_size) for CTC loss
        """
        # Pack for efficient computation
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)  # (B, T, H*2)
        
        out = self.dropout(out)
        logits = self.fc(out)  # (B, T, vocab_size)
        
        # CTC expects (T, B, C)
        log_probs = torch.log_softmax(logits, dim=-1).permute(1, 0, 2)
        
        return log_probs

# Model factory for different experiments
def create_model(input_dim, vocab_size, hidden_dim=256, num_layers=3, dropout=0.3):
    model = BiLSTM_CTC(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
        vocab_size=vocab_size
    )
    return model.to(device)

print("BiLSTM-CTC model defined")

In [ ]:
# ============================================================
# CTC Decoding & CER Evaluation
# ============================================================

def greedy_decode(log_probs, lengths, idx2char):
    """
    Greedy CTC decoding.
    
    Args:
        log_probs: (T, B, vocab_size)
        lengths: (B,) actual output lengths
        idx2char: index to character mapping
    Returns:
        decoded strings
    """
    # Get argmax predictions
    predictions = log_probs.argmax(dim=-1).T  # (B, T)
    
    decoded = []
    for i, pred in enumerate(predictions):
        # Trim to actual length
        pred = pred[:lengths[i]].tolist()
        
        # Collapse repeats and remove blanks
        chars = []
        prev = None
        for p in pred:
            if p != prev and p != 0:  # 0 is blank
                chars.append(idx2char.get(p, '?'))
            prev = p
        decoded.append(''.join(chars))
    
    return decoded

def compute_cer(predictions, targets):
    """
    Compute Character Error Rate using edit distance.
    Also returns insertion/deletion/substitution counts.
    """
    total_chars = 0
    total_edits = 0
    total_insertions = 0
    total_deletions = 0
    total_substitutions = 0
    
    for pred, target in zip(predictions, targets):
        # Compute edit distance with backtracking
        n, m = len(target), len(pred)
        dp = [[0] * (m + 1) for _ in range(n + 1)]
        
        for i in range(n + 1):
            dp[i][0] = i
        for j in range(m + 1):
            dp[0][j] = j
        
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                if target[i-1] == pred[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        
        # Backtrack to count I/D/S
        i, j = n, m
        ins, dels, subs = 0, 0, 0
        while i > 0 or j > 0:
            if i > 0 and j > 0 and target[i-1] == pred[j-1]:
                i -= 1
                j -= 1
            elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
                subs += 1
                i -= 1
                j -= 1
            elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
                ins += 1
                j -= 1
            else:
                dels += 1
                i -= 1
        
        total_chars += len(target)
        total_edits += dp[n][m]
        total_insertions += ins
        total_deletions += dels
        total_substitutions += subs
    
    cer = total_edits / max(total_chars, 1)
    return {
        'cer': cer,
        'insertions': total_insertions,
        'deletions': total_deletions,
        'substitutions': total_substitutions,
        'total_chars': total_chars,
        'total_edits': total_edits
    }

print("CER evaluation functions defined")

In [ ]:
# ============================================================
# Training Function
# ============================================================

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, experiment_name="exp"):
    """Train BiLSTM-CTC model."""
    
    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_cer = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_cer': []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for batch in train_loader:
            feats, targets, feat_lengths, target_lengths, _ = batch
            feats = feats.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            log_probs = model(feats, feat_lengths)
            
            # CTC loss expects (T, B, C), targets (B, S)
            # Input lengths after LSTM (same as feat_lengths for our model)
            loss = ctc_loss(log_probs, targets, feat_lengths, target_lengths)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch in val_loader:
                feats, targets, feat_lengths, target_lengths, _ = batch
                feats = feats.to(device)
                targets = targets.to(device)
                
                log_probs = model(feats, feat_lengths)
                loss = ctc_loss(log_probs, targets, feat_lengths, target_lengths)
                val_loss += loss.item()
                
                # Decode predictions
                decoded = greedy_decode(log_probs.cpu(), feat_lengths, idx2char)
                all_preds.extend(decoded)
                
                # Get target strings
                for i in range(len(targets)):
                    target_str = ''.join([idx2char.get(t.item(), '?') for t in targets[i][:target_lengths[i]]])
                    all_targets.append(target_str)
        
        val_loss /= len(val_loader)
        cer_result = compute_cer(all_preds, all_targets)
        val_cer = cer_result['cer']
        
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_cer'].append(val_cer)
        
        if val_cer < best_cer:
            best_cer = val_cer
            torch.save(model.state_dict(), f"models/{experiment_name}_best.pt")
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val CER: {val_cer:.4f} | "
                  f"I/D/S: {cer_result['insertions']}/{cer_result['deletions']}/{cer_result['substitutions']}")
    
    return history, best_cer

# Create models directory
Path("models").mkdir(exist_ok=True)
print("Training function defined")

In [ ]:
# ============================================================
# Run Experiments: Compare Features
# ============================================================
# Primary: 80-dim Log-Mel + Global CMVN
# Compare: 40-dim MFCC, Log-Mel + Delta
# ============================================================

from sklearn.model_selection import train_test_split

# Split data
train_items, val_items = train_test_split(train_data, test_size=0.1, random_state=42)
print(f"Train: {len(train_items)}, Val: {len(val_items)}")

# Experiment configurations
experiments = {
    'logmel80': {
        'feature_dir': features_dir / 'exp1_logmel80',
        'input_dim': 80,
        'description': '80-dim Log-Mel (Primary)'
    },
    'mfcc40': {
        'feature_dir': features_dir / 'exp1_mfcc40',
        'input_dim': 40,
        'description': '40-dim MFCC'
    },
    'logmel_delta': {
        'feature_dir': features_dir / 'exp2_logmel_delta',
        'input_dim': 240,  # 80 + 80 + 80
        'description': 'Log-Mel + Δ + ΔΔ (240-dim)'
    }
}

# Store results
results = {}

In [ ]:
# ============================================================
# Train All Experiments
# ============================================================

BATCH_SIZE = 16
EPOCHS = 50
HIDDEN_DIM = 256
NUM_LAYERS = 3
DROPOUT = 0.3
LR = 1e-3

for exp_name, config in experiments.items():
    print(f"\n{'='*60}")
    print(f"Experiment: {config['description']}")
    print(f"{'='*60}")
    
    # Create datasets
    train_dataset = SpeechDataset(train_items, config['feature_dir'], char2idx)
    val_dataset = SpeechDataset(val_items, config['feature_dir'], char2idx)
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print(f"Skipping {exp_name}: no data found")
        continue
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                              collate_fn=collate_fn, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=0)
    
    # Create model
    model = create_model(
        input_dim=config['input_dim'],
        vocab_size=vocab_size,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    )
    
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
    
    # Train
    history, best_cer = train_model(
        model, train_loader, val_loader,
        epochs=EPOCHS, lr=LR, experiment_name=exp_name
    )
    
    results[exp_name] = {
        'best_cer': best_cer,
        'history': history,
        'config': config
    }
    
    print(f"\n{exp_name} Best CER: {best_cer:.4f}")

In [ ]:
# ============================================================
# Results Analysis & Confusion Patterns
# ============================================================
import matplotlib.pyplot as plt
from collections import Counter

def analyze_confusions(model, val_loader, idx2char, char2idx, top_k=20):
    """Analyze character confusion patterns."""
    model.eval()
    confusions = Counter()
    
    with torch.no_grad():
        for batch in val_loader:
            feats, targets, feat_lengths, target_lengths, _ = batch
            feats = feats.to(device)
            
            log_probs = model(feats, feat_lengths)
            decoded = greedy_decode(log_probs.cpu(), feat_lengths, idx2char)
            
            for i in range(len(targets)):
                target_str = ''.join([idx2char.get(t.item(), '?') for t in targets[i][:target_lengths[i]]])
                pred_str = decoded[i]
                
                # Align and find confusions (simplified)
                for j in range(min(len(target_str), len(pred_str))):
                    if target_str[j] != pred_str[j]:
                        confusions[(target_str[j], pred_str[j])] += 1
    
    print(f"\nTop {top_k} Character Confusions (target → prediction):")
    for (t, p), count in confusions.most_common(top_k):
        print(f"  '{t}' → '{p}': {count}")
    
    return confusions

# Plot results comparison
if results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # CER comparison
    exp_names = list(results.keys())
    cers = [results[e]['best_cer'] for e in exp_names]
    axes[0].bar(exp_names, cers, color=['#2ecc71', '#3498db', '#e74c3c'])
    axes[0].set_ylabel('CER')
    axes[0].set_title('Best CER by Feature Type')
    axes[0].set_ylim(0, max(cers) * 1.2)
    for i, v in enumerate(cers):
        axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')
    
    # Training curves
    for exp_name, data in results.items():
        axes[1].plot(data['history']['train_loss'], label=f'{exp_name} (train)')
        axes[1].plot(data['history']['val_loss'], '--', label=f'{exp_name} (val)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training Curves')
    axes[1].legend()
    
    # CER curves
    for exp_name, data in results.items():
        axes[2].plot(data['history']['val_cer'], label=exp_name)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('CER')
    axes[2].set_title('Validation CER')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('feature_comparison.png', dpi=150)
    plt.show()
    
    # Summary table
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(f"{'Experiment':<20} {'Input Dim':<12} {'Best CER':<10}")
    print("-"*60)
    for exp_name, data in results.items():
        print(f"{exp_name:<20} {data['config']['input_dim']:<12} {data['best_cer']:.4f}")
else:
    print("No results to display. Run experiments first.")

In [ ]:
# ============================================================
# Detailed Analysis: Primary Model (Log-Mel 80)
# ============================================================

if 'logmel80' in results:
    # Load best model
    best_model = create_model(input_dim=80, vocab_size=vocab_size)
    best_model.load_state_dict(torch.load("models/logmel80_best.pt"))
    
    # Recreate val loader
    val_dataset = SpeechDataset(val_items, features_dir / 'exp1_logmel80', char2idx)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    # Analyze confusions
    confusions = analyze_confusions(best_model, val_loader, idx2char, char2idx)
    
    # Final CER breakdown
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            feats, targets, feat_lengths, target_lengths, _ = batch
            feats = feats.to(device)
            log_probs = best_model(feats, feat_lengths)
            decoded = greedy_decode(log_probs.cpu(), feat_lengths, idx2char)
            all_preds.extend(decoded)
            for i in range(len(targets)):
                target_str = ''.join([idx2char.get(t.item(), '?') for t in targets[i][:target_lengths[i]]])
                all_targets.append(target_str)
    
    final_cer = compute_cer(all_preds, all_targets)
    
    print("\n" + "="*60)
    print("PRIMARY MODEL (Log-Mel 80) - ERROR BREAKDOWN")
    print("="*60)
    print(f"CER: {final_cer['cer']:.4f}")
    print(f"Insertions:    {final_cer['insertions']:5d} ({final_cer['insertions']/final_cer['total_edits']*100:.1f}%)")
    print(f"Deletions:     {final_cer['deletions']:5d} ({final_cer['deletions']/final_cer['total_edits']*100:.1f}%)")
    print(f"Substitutions: {final_cer['substitutions']:5d} ({final_cer['substitutions']/final_cer['total_edits']*100:.1f}%)")
    print(f"Total Edits:   {final_cer['total_edits']}")
    print(f"Total Chars:   {final_cer['total_chars']}")
else:
    print("Run experiments first to see detailed analysis.")